# Etapa 4 — Modelagem

**Objetivo:** Treinar 3 modelos diferentes e salvá-los para avaliação.

Os três modelos que vamos comparar:

| Modelo | Analogia simples |
|--------|------------------|
| **Random Forest** | Um grupo de detetives votando — cada um vê parte dos dados |
| **XGBoost** | Um detetive que aprende com os próprios erros, iterativamente |
| **MLP (Rede Neural)** | Uma rede de 'neurônios' artificiais que aprendem padrões complexos |

Todos são treinados na mesma tabela de features (saída da Etapa 3) para comparação justa.

**Importante:** o MLP recebe os dados normalizados (StandardScaler), pois redes neurais são
sensíveis à escala dos números. Árvores não precisam disso.

In [1]:
import sys
sys.path.insert(0, '..')

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

from config import (
    FEATURES_DATA_PATH,
    MODELS_DIR,
    N_ITER_SEARCH,
    N_JOBS,
    N_SPLITS_CV,
    RANDOM_STATE,
    VALIDATION_MODE,
)

mode_label = 'VALIDACAO' if VALIDATION_MODE else 'COMPLETO'
print(f"{'='*55}")
print(f"  Modo: {mode_label}")
print(f"  Folds CV : {N_SPLITS_CV}  |  Iteracoes busca: {N_ITER_SEARCH}  |  Jobs: {N_JOBS}")
if VALIDATION_MODE:
    print("  Grids de hiperparametros reduzidos para velocidade.")
    print("  Mude VALIDATION_MODE = False para busca completa.")
print(f"{'='*55}")

ModuleNotFoundError: No module named 'xgboost'

## 4.1 Preparar dados para modelagem

In [ ]:
df = pd.read_parquet(FEATURES_DATA_PATH)

META_COLS = ['instance_id', 'fault_class', 'source_type', 'window_start']
feature_cols = [c for c in df.columns if c not in META_COLS]

X = df[feature_cols].values
y = df['fault_class'].values
groups = df['instance_id'].values  # usado no GroupKFold

# Preencher NaN remanescentes com mediana (janelas muito curtas)
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='median')
X = imputer.fit_transform(X)

print(f'Shape X: {X.shape}')
print(f'Classes: {np.unique(y)}')
print(f'Instâncias únicas (grupos): {len(np.unique(groups))}')

## 4.2 Definir modelos e grids de hiperparâmetros

O `RandomizedSearchCV` testa `N_ITER_SEARCH` combinações aleatórias e escolhe a melhor —
muito mais rápido do que testar todas as combinações (GridSearch).

No **modo validação**, os grids são menores (menos opções) e `N_ITER_SEARCH=5`.
No **modo completo**, os grids são maiores e `N_ITER_SEARCH=20`.

In [ ]:
gkf = GroupKFold(n_splits=N_SPLITS_CV)

# Grids adaptados ao modo de execução
if VALIDATION_MODE:
    # Grids pequenos: rápido para validar que o código funciona
    rf_param_grid = {
        'n_estimators': [50, 100],
        'max_depth': [None, 10],
        'min_samples_leaf': [1, 2],
        'max_features': ['sqrt'],
    }
    xgb_param_grid = {
        'n_estimators': [50, 100],
        'max_depth': [3, 5],
        'learning_rate': [0.1, 0.2],
        'subsample': [0.9, 1.0],
    }
    mlp_param_grid = {
        'mlp__hidden_layer_sizes': [(64, 32), (128, 64)],
        'mlp__alpha': [1e-4, 1e-3],
        'mlp__learning_rate_init': [1e-3],
    }
else:
    # Grids completos: busca mais ampla de hiperparâmetros
    rf_param_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [None, 10, 20, 30],
        'min_samples_leaf': [1, 2, 4],
        'max_features': ['sqrt', 'log2'],
    }
    xgb_param_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [3, 5, 7],
        'learning_rate': [0.05, 0.1, 0.2],
        'subsample': [0.7, 0.9, 1.0],
        'colsample_bytree': [0.7, 0.9, 1.0],
    }
    mlp_param_grid = {
        'mlp__hidden_layer_sizes': [(128, 64), (256, 128), (128, 64, 32)],
        'mlp__alpha': [1e-4, 1e-3, 1e-2],
        'mlp__learning_rate_init': [1e-3, 5e-4],
    }

# ── Instanciar modelos ────────────────────────────────────────────────────────
rf_model = RandomForestClassifier(
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS,
)

xgb_model = XGBClassifier(
    objective='multi:softmax',
    num_class=10,
    eval_metric='mlogloss',
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS,
    verbosity=0,
)

# MLP exige StandardScaler — redes neurais são sensíveis à escala dos dados
mlp_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPClassifier(
        max_iter=300,
        random_state=RANDOM_STATE,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=15,
    )),
])

MODELS = {
    'random_forest': (rf_model, rf_param_grid),
    'xgboost':       (xgb_model, xgb_param_grid),
    'mlp':           (mlp_pipeline, mlp_param_grid),
}

print(f'{len(MODELS)} modelos configurados para o modo {mode_label}:'
      f' {list(MODELS.keys())}')

## 4.3 Treinar e salvar os modelos

Para cada modelo: buscamos os melhores hiperparâmetros com cross-validation
e salvamos o modelo final em `results/models/`.

In [ ]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)
trained_models = {}

for model_name, (estimator, param_grid) in MODELS.items():
    print(f'\n{'─'*50}')
    print(f'Treinando: {model_name}')

    search = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=param_grid,
        n_iter=N_ITER_SEARCH,
        cv=gkf,
        scoring='f1_macro',
        n_jobs=N_JOBS,
        random_state=RANDOM_STATE,
        verbose=1,
        refit=True,
    )

    search.fit(X, y, groups=groups)

    print(f'Melhor F1-macro (CV): {search.best_score_:.4f}')
    print(f'Melhores parâmetros: {search.best_params_}')

    # Salvar modelo
    model_path = MODELS_DIR / f'{model_name}.joblib'
    joblib.dump(search.best_estimator_, model_path)
    print(f'Salvo em: {model_path}')

    trained_models[model_name] = search.best_estimator_

print('\nTodos os modelos treinados e salvos!')